<a href="https://colab.research.google.com/github/Joan2022Laurente/fade/blob/main/notebooks/zImageTurboV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
#   SETUP — Dark Beast (Z-Image Turbo) + Qwen3-4B + VAE + LoRA
#   + modelos custom con barra de progreso
# ================================================================

# ── FORMULARIO ─────────────────────────────────────────────────

# --- Checkpoint 1 ---
CKPT1_URL  = "https://civitai.red/api/download/models/2957651?fileId=2837007"  # @param {type:"string"}
CKPT1_NAME = "divingZimageTurbo.safetensors"  # @param {type:"string"}

# --- Checkpoint 2 ---
CKPT2_URL  = "https://civitai.red/api/download/models/2918406?fileId=2804511"                           # @param {type:"string"}
CKPT2_NAME = "prnmasterBASE.safetensors"        # @param {type:"string"}

# --- Checkpoint 3 ---
CKPT3_URL  = "https://civitai.red/api/download/models/2754977?fileId=2641972"                           # @param {type:"string"}
CKPT3_NAME = "RedCraftERNIERedMIX.safetensors"        # @param {type:"string"}

# --- Checkpoint 4 ---
CKPT4_URL  = "https://civitai.red/api/download/models/2668773?fileId=2555711"                           # @param {type:"string"}
CKPT4_NAME = "byStable_Yogi.safetensors"        # @param {type:"string"}

# --- LoRA HuggingFace (fijo) ---
LORA_HF_ENABLED = False                    # @param {type:"boolean"}

# --- LoRA Civitai 1 ---
LORA1_URL  = ""                           # @param {type:"string"}
LORA1_NAME = "zitMystic.safetensors"          # @param {type:"string"}

# --- LoRA Civitai 2 ---
LORA2_URL  = "https://civitai.red/api/download/models/2486059?fileId=2374406"                           # @param {type:"string"}
LORA2_NAME = "nsMASTER.safetensors"          # @param {type:"string"}

# --- LoRA Civitai 3 ---
LORA3_URL  = "https://civitai.red/api/download/models/2454516?fileId=2344803"                           # @param {type:"string"}
LORA3_NAME = "povor4.safetensors"          # @param {type:"string"}

# --- LoRA Civitai 4 ---
LORA4_URL  = "https://civitai.red/api/download/models/2643568?fileId=2531431"                           # @param {type:"string"}
LORA4_NAME = "ahe.safetensors"          # @param {type:"string"}

# ───────────────────────────────────────────────────────────────
import os, subprocess, sys, shutil, re
from google.colab import userdata
from tqdm.notebook import tqdm

CIVITAI_KEY   = userdata.get('CIVITAI_KEY')
TAILSCALE_KEY = userdata.get('TAILSCALE_KEY')
HF_TOKEN      = userdata.get('HF_TOKEN')
if not CIVITAI_KEY:   raise RuntimeError("❌ Falta CIVITAI_KEY")
if not TAILSCALE_KEY: raise RuntimeError("❌ Falta TAILSCALE_KEY")
if not HF_TOKEN:      raise RuntimeError("❌ Falta HF_TOKEN")
print("✅ Secrets cargados")

COMFY        = "/content/ComfyUI"
MODELS       = f"{COMFY}/models"
CUSTOM_NODES = f"{COMFY}/custom_nodes"
WORKFLOWS    = f"{COMFY}/user/default/workflows"
HF_Z         = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files"

# ── Helpers base ───────────────────────────────────────────────
def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout.strip(): print(r.stdout.strip()[-1500:])
    return r.returncode

def pip(pkg):
    run(f'"{sys.executable}" -m pip install {pkg} -q --no-deps 2>/dev/null || '
        f'"{sys.executable}" -m pip install {pkg} -q')

def clone(url, dst):
    if os.path.isdir(dst): run("git pull --ff-only", cwd=dst)
    else: run(f'git clone --depth 1 "{url}" "{dst}"')
    req = f"{dst}/requirements.txt"
    if os.path.exists(req): run(f'"{sys.executable}" -m pip install -r "{req}" -q')

# ── Descarga con barra de progreso ─────────────────────────────
def dl_with_progress(url, folder, name, auth_header=None):
    os.makedirs(folder, exist_ok=True)
    dest = f"{folder}/{name}"

    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        print(f"  ✅ {name} ya existe — skip")
        return dest

    auth = f'--header "Authorization: Bearer {auth_header}"' if auth_header else ""
    cmd  = f'wget -L --progress=dot:mega {auth} -O "{dest}" "{url}"'

    bar  = tqdm(total=None, unit='B', unit_scale=True, unit_divisor=1024,
                desc=f"  ↓ {name}", dynamic_ncols=True, colour="cyan")

    proc = subprocess.Popen(cmd, shell=True, stderr=subprocess.PIPE,
                            stdout=subprocess.DEVNULL, text=True, bufsize=1)
    last = 0
    for line in proc.stderr:
        m = re.search(r'([\d,.]+[KMG]?)\s+(\d+)%', line)
        if m:
            pct = int(m.group(2))
            if bar.total is None:
                raw = m.group(1).replace(',', '')
                mul = {'K': 1024, 'M': 1024**2, 'G': 1024**3}
                factor = mul.get(raw[-1], 1)
                try:
                    current_bytes = float(re.sub(r'[KMG]', '', raw)) * factor
                    if pct > 0:
                        bar.total = int(current_bytes * 100 / pct)
                except ValueError:
                    pass
            if bar.total:
                new = int(bar.total * pct / 100)
                bar.update(new - last)
                last = new

    proc.wait()
    bar.close()

    ok = os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024
    sz = os.path.getsize(dest) / 1024**3 if ok else 0
    print(f"  {'✅' if ok else '❌'} {name} ({sz:.2f} GB)")
    return dest if ok else None

def dl_hf(url, folder, name, token=None):
    return dl_with_progress(url, folder, name, auth_header=token)

def dl_civitai(url, folder, name):
    sep      = "&" if "?" in url else "?"
    full_url = f"{url}{sep}token={CIVITAI_KEY}"
    return dl_with_progress(full_url, folder, name)

# ── SISTEMA ────────────────────────────────────────────────────
print("\n[SYS] Herramientas...")
run("apt-get update -qq && apt-get install -y -qq aria2 git wget curl")

# ── COMFYUI ────────────────────────────────────────────────────
print("\n[0] ComfyUI...")
if not os.path.isdir(COMFY):
    run(f'git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY}"')
else:
    run("git pull --ff-only", cwd=COMFY)
run(f'"{sys.executable}" -m pip install -r "{COMFY}/requirements.txt" -q')
for d in [f"{MODELS}/diffusion_models", f"{MODELS}/text_encoders",
          f"{MODELS}/vae", f"{MODELS}/loras", CUSTOM_NODES, WORKFLOWS]:
    os.makedirs(d, exist_ok=True)

# ── CUSTOM NODES ───────────────────────────────────────────────
print("\n[1] rgthree-comfy...")
clone("https://github.com/rgthree/rgthree-comfy.git", f"{CUSTOM_NODES}/rgthree-comfy")

print("\n[1.5] azzia-nodes (Custom)...")
clone("https://github.com/Joan2022Laurente/node.git", f"{CUSTOM_NODES}/azzia-nodes")

# ── DEPS ───────────────────────────────────────────────────────
print("\n[2] Deps...")
for p in ["accelerate", "einops", "torchvision", "Pillow"]: pip(p)
print("  ✅")

# ── MODELOS FIJOS (encoder + VAE) ─────────────────────────────
print("\n[3] Text encoder + VAE...")
dl_hf(f"{HF_Z}/text_encoders/qwen_3_4b.safetensors",
      f"{MODELS}/text_encoders", "qwen_3_4b.safetensors", token=HF_TOKEN)
dl_hf(f"{HF_Z}/vae/ae.safetensors",
      f"{MODELS}/vae", "ae.safetensors", token=HF_TOKEN)

# ── CHECKPOINTS (Civitai → diffusion_models) ──────────────────
print("\n[4] Checkpoints...")
CHECKPOINTS = [
    (CKPT1_URL, CKPT1_NAME),
    (CKPT2_URL, CKPT2_NAME),
    (CKPT3_URL, CKPT3_NAME),
    (CKPT4_URL, CKPT4_NAME),
]
for i, (url, name) in enumerate(CHECKPOINTS, 1):
    if url.strip():
        print(f"  → Checkpoint {i}: {name}")
        dl_civitai(url.strip(), f"{MODELS}/diffusion_models", name.strip())
    else:
        print(f"  ⚪ Checkpoint {i}: vacío — skip")

# ── LoRAs ─────────────────────────────────────────────────────
print("\n[5] LoRAs...")

if LORA_HF_ENABLED:
    print("  → LoRA HF: YummyHDzit.safetensors")
    dl_hf("https://huggingface.co/joanjlau/loraprueba0/resolve/main/YummyHDzit.safetensors",
          f"{MODELS}/loras", "YummyHDzit.safetensors", token=HF_TOKEN)
else:
    print("  ⚪ LoRA HF: desactivado — skip")

LORAS = [
    (LORA1_URL, LORA1_NAME),
    (LORA2_URL, LORA2_NAME),
    (LORA3_URL, LORA3_NAME),
    (LORA4_URL, LORA4_NAME),
]
for i, (url, name) in enumerate(LORAS, 1):
    if url.strip():
        print(f"  → LoRA Civitai {i}: {name}")
        dl_civitai(url.strip(), f"{MODELS}/loras", name.strip())
    else:
        print(f"  ⚪ LoRA Civitai {i}: vacío — skip")

# ── WORKFLOW ───────────────────────────────────────────────────
print("\n[6] Workflow...")
src = f"{CUSTOM_NODES}/azzia-nodes/ZIMAGET2I_lora.json"
if os.path.exists(src):
    shutil.copy2(src, f"{WORKFLOWS}/ZIMAGET2I_lora.json")
    print("  ✅ Workflow copiado automáticamente desde azzia-nodes")
else:
    print("  ⚠️ No se encontró el workflow en azzia-nodes")

# ── VERIFICACIÓN ───────────────────────────────────────────────
print("\n=== VERIFICACIÓN ===")

print("\n  — Fijos —")
for path, label in [
    (f"{MODELS}/text_encoders/qwen_3_4b.safetensors", "Qwen3-4B encoder"),
    (f"{MODELS}/vae/ae.safetensors",                  "Flux VAE"),
    (f"{CUSTOM_NODES}/rgthree-comfy",                 "rgthree"),
    (f"{CUSTOM_NODES}/azzia-nodes",                   "azzia-nodes"),
]:
    ok = os.path.exists(path) and (not os.path.isfile(path) or os.path.getsize(path) > 1024*1024)
    print(f"  {'✅' if ok else '❌'} {label}")

print("\n  — Checkpoints —")
for i, (url, name) in enumerate(CHECKPOINTS, 1):
    if url.strip():
        path = f"{MODELS}/diffusion_models/{name}"
        ok   = os.path.exists(path) and os.path.getsize(path) > 1024*1024
        print(f"  {'✅' if ok else '❌'} Checkpoint {i}: {name}")
    else:
        print(f"  ⚪ Checkpoint {i}: no configurado")

print("\n  — LoRAs —")
if LORA_HF_ENABLED:
    path = f"{MODELS}/loras/YummyHDzit.safetensors"
    ok   = os.path.exists(path) and os.path.getsize(path) > 1024*1024
    print(f"  {'✅' if ok else '❌'} LoRA HF: YummyHDzit.safetensors")
else:
    print("  ⚪ LoRA HF: desactivado")

for i, (url, name) in enumerate(LORAS, 1):
    if url.strip():
        path = f"{MODELS}/loras/{name}"
        ok   = os.path.exists(path) and os.path.getsize(path) > 1024*1024
        print(f"  {'✅' if ok else '❌'} LoRA Civitai {i}: {name}")
    else:
        print(f"  ⚪ LoRA Civitai {i}: no configurado")

print("\n✅ SETUP COMPLETO")
print("→ KSampler: Steps=8 · CFG=1.0 · euler · beta")


✅ Secrets cargados

[SYS] Herramientas...
:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is n

  ↓ qwen_3_4b.safetensors: 0.00B [00:00, ?B/s]

  ✅ qwen_3_4b.safetensors (7.49 GB)


  ↓ ae.safetensors: 0.00B [00:00, ?B/s]

  ✅ ae.safetensors (0.31 GB)

[4] Checkpoints...
  → Checkpoint 1: divingZimageTurbo.safetensors


  ↓ divingZimageTurbo.safetensors: 0.00B [00:00, ?B/s]

In [ ]:
# 🔒 Lanzar ComfyUI con Tailscale

import os, time
from google.colab import userdata

TAILSCALE_AUTH_KEY = userdata.get('TAILSCALE_KEY')
if not TAILSCALE_AUTH_KEY:
    raise RuntimeError("❌ Falta TAILSCALE_KEY en Secrets de Colab")

# 1. Instalar Tailscale
!curl -fsSL https://tailscale.com/install.sh | sh

# 2. Iniciar daemon
print("🚀 Iniciando tailscaled...")
!nohup tailscaled --tun=userspace-networking --socks5-server=localhost:1055 > /tmp/tailscaled.log 2>&1 &
time.sleep(5)

# 3. Verificar
print("🔍 Verificando tailscaled...")
!ps aux | grep tailscaled

# 4. Conectar
print("🔗 Conectando a Tailscale...")
!tailscale up --authkey={TAILSCALE_AUTH_KEY} --hostname=colab-comfyui

# 5. Mostrar IP
print("\n" + "="*50)
print("🌐 TU IP PRIVADA DE TAILSCALE ES:")
!tailscale ip -4
print("="*50)

# 6. Lanzar ComfyUI
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n🚀 Iniciando ComfyUI...")
%cd /content/ComfyUI
!python main.py --listen 0.0.0.0 --dont-print-server

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<script>
const ctx = new (window.AudioContext || window.webkitAudioContext)();
const merger = ctx.createChannelMerger(2);

const gainL = ctx.createGain();
const gainR = ctx.createGain();
gainL.gain.value = 0.4;
gainR.gain.value = 0.4;

const oscL = ctx.createOscillator();
const oscR = ctx.createOscillator();
oscL.type = 'sine';
oscR.type = 'sine';
oscL.frequency.value = 100;   // oído izquierdo
oscR.frequency.value = 133;   // oído derecho (diferencia = 33 Hz Gamma)

oscL.connect(gainL); gainL.connect(merger, 0, 0);
oscR.connect(gainR); gainR.connect(merger, 0, 1);
merger.connect(ctx.destination);

oscL.start();
oscR.start();
</script>
<small style="color:gray">🎧 Binaural activo — 100 Hz / 133 Hz (Gamma 33 Hz)</small>
"""))
# =========================================================
# COMFYUI + PINGGY TUNNEL
# =========================================================

import subprocess
import threading
import time
import re

# =========================================================
# FUNCIÓN TÚNEL PINGGY
# =========================================================

def run_pinggy():
    print("🌐 Iniciando túnel Pinggy...")

    process = subprocess.Popen(
        [
            "ssh",
            "-p", "443",
            "-o", "StrictHostKeyChecking=no",
            "-R0:localhost:8188",
            "a.pinggy.io"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line.strip())

        match = re.search(r"https://[^\s]+\.a\.free\.pinggy\.link", line)

        if match:
            print("\n" + "=" * 70)
            print("🚀 COMFYUI PUBLIC URL:")
            print(match.group(0))
            print("=" * 70 + "\n")

# =========================================================
# LANZAR TÚNEL
# =========================================================

threading.Thread(
    target=run_pinggy,
    daemon=True
).start()

# =========================================================
# ESPERA
# =========================================================

time.sleep(5)

# =========================================================
# INICIAR COMFYUI
# =========================================================

%cd /content/ComfyUI

!python main.py \
    --listen 0.0.0.0 \
    --port 8188 \
    --dont-print-server

